In [ ]:
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
ontology = pd.read_csv("../1_enhance_UMLS/04_ConceptDB/umls-dutch_v1.11_with_drugs_filtered-categories.csv")

ontology["cui"] = ontology["cui"].astype(str)
ontology["name"] = ontology["name"].astype(str)

cui2names = ontology.groupby("cui")["name"].apply(list).to_dict()
valid_cuis = set(cui2names.keys())

walvis = pd.read_csv("../2_generate_corpus/walvis_nl_cat4.csv")
# walvis = pd.read_csv("../2_generate_corpus/walvis_en_cat4.csv")
walvis["cui"] = walvis["cui"].astype(str)

In [ ]:
print(len(walvis))
print(len(walvis["cui"].unique()))

In [ ]:
walvis_star = walvis[walvis["cui"].isin(valid_cuis)]
walvis_star = walvis_star.drop_duplicates(subset=["mention"])
print(len(walvis_star))
print(len(walvis_star["cui"].unique()))
walvis_star.to_csv("walvis_star_nl.csv", index=False)
# walvis_star.to_csv("walvis_star_en.csv", index=False)

In [ ]:
finetune_lines = []

for _, row in tqdm(walvis_star.iterrows(), total=len(walvis_star)):

    cui = row["cui"]
    mention = str(row["mention"]).strip()
    term = cui2names[cui][0].strip()

    if mention and term:
        finetune_lines.append(f"{cui}||{mention}||{term}")

In [ ]:
# with open("en_pairs.txt", "w", encoding='utf-8') as f:
with open("nl_pairs.txt", "w", encoding='utf-8') as f:
    for line in finetune_lines:
        f.write(line + "\n")

If you want to use the English pairs, first run the code above but then with the commented out English parts instead of the Dutch parts. Next, use the notebook `translate_pairs.ipynb` for translating `en_pairs.txt`. Then continue with code below.

Add the English translations

In [ ]:
input_file = "en_translated_pairs.txt"
with open(input_file, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]


data = []
english_texts = []
for line in lines:
    parts = [p.strip() for p in line.split("||")]
    cui = parts[0]
    english = parts[1]
    english.strip(".")
    # filter out bad translations
    if "is" in english:
        continue
    if english.lower().startswith("het ") or english.lower().startswith("een "):
        english = english[4:]
    if english.lower().startswith("de "):
        english = english[3:]
    seen = set()
    clean_words = []

    for w in english.split():
        if w.lower() not in seen:
            seen.add(w.lower())
            clean_words.append(w.lower())

    english = " ".join(clean_words)

    tail = parts[2:]
    data.append({"CUI": cui, "English": english, "Tail": tail})

In [ ]:
output_lines = []
output_file = "en_translated_pairs.txt"
for entry in data:
    cui = entry["CUI"]
    english = entry["English"]
    tail = entry["Tail"]
    output_lines.append("||".join([cui, english] + tail))

with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

Combine Dutch and English rows

In [ ]:
df1 = pd.read_csv("nl_pairs.txt", sep=r"\|\|", header=None)
df2 = pd.read_csv("en_translated_pairs.txt", sep=r"\|\|", header=None)

combined = pd.concat([df1, df2], ignore_index=True)

# remove duplicate rows (keep first occurrence)
deduplicated = combined.drop_duplicates(keep="first")

# save result
with open("combined_pairs.txt", "w", encoding="utf-8") as f:
    for row in deduplicated.itertuples(index=False, name=None):
        f.write("||".join(map(str, row)) + "\n")